# 01 - Data cleaning & target construction

**What Makes a Song Sticky?** This notebook loads Spotify track-level audio features and prepares analysis-ready tables.

**Framing:** We are *not* measuring addiction, replays, skips, or saves. **Popularity** is a public **proxy** for broad replayability. **Stickiness** here means: tracks in the **top ~20%** of popularity in *this* dataset (at or above the 80th percentile).

**Working directory:** Run Jupyter from the repo root, or from `notebooks/` - the next cell resolves `PROJECT_ROOT` automatically.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import data_prep

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "spotify_tracks.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CLEAN_PATH = PROCESSED_DIR / "spotify_cleaned.csv"
MODEL_PATH = PROCESSED_DIR / "spotify_model_data.csv"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


## 1. Load raw data


In [ ]:
df_raw = data_prep.load_data(RAW_PATH)


## 2. Inspect: columns, head, info, missingness


In [ ]:
print("Column names:", list(df_raw.columns))
print("Shape:", df_raw.shape)
display(df_raw.head())
df_raw.info()
miss = pd.DataFrame({
    "missing": df_raw.isna().sum(),
    "pct": (df_raw.isna().mean() * 100).round(2),
})
display(miss[miss["missing"] > 0] if miss["missing"].sum() else miss.head(0))
print("Missing summary (all columns):")
display(miss)


## 3. Cleaning pipeline (automated)

The module [`src/data_prep.py`](../src/data_prep.py) applies, in order:

1. **Column mapping** - `COLUMN_MAP` maps common alternate names (e.g. `artists` → `artist_name`, `duration` → `duration_ms`) to canonical names.
2. **Duration** - `duration_sec` or values that look like **seconds** are converted to milliseconds when needed.
3. **Duplicates** - dropped on `track_name` + `artist_name` + `duration_ms` when those columns exist.
4. **Missing values** - rows with missing **popularity** or any core **audio feature** are dropped (optional fields like `genre` are not required).
5. **Validation** - popularity in [0, 100]; Spotify features in [0, 1]; `duration_ms` > 0.
6. **Targets** - `popularity_z` (z-score); `sticky` = 1 if popularity ≥ 80th percentile (top ~20%), else 0.


In [ ]:
df, stats = data_prep.clean_dataframe(df_raw)
print("Sticky threshold (80th percentile):", stats.get("sticky_threshold"))
print("Class balance (proportion sticky):", round(stats.get("sticky_balance", 0), 4))
print("Rows after cleaning:", stats.get("n_rows_final"))
print("Duplicates removed:", stats.get("duplicates_removed"))
print("Validation notes:", stats.get("validation_notes"))


## 4. Summary statistics (cleaned numeric columns)


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
display(df[num_cols].describe().T)


## 5. Save processed datasets


In [ ]:
keep_meta = [c for c in ("track_name", "artist_name", "genre") if c in df.columns]
keep_num = [c for c in data_prep.EXPECTED_COLUMNS if c in df.columns and c not in keep_meta]
extra = ["popularity_z", "sticky"]
all_cols = keep_meta + [c for c in keep_num if c not in keep_meta] + [e for e in extra if e in df.columns]
df_out = df[all_cols].copy()

data_prep.save_processed_data(df_out, CLEAN_PATH)
data_prep.save_processed_data(df_out, MODEL_PATH)
print("Saved:", CLEAN_PATH)
print("Saved:", MODEL_PATH)


## 6. Next steps

Run **`02_eda.ipynb`** on `spotify_model_data.csv`, then **`03_modeling.ipynb`**.
